# Music Annotation Project V8.7
## Notebook 49: Residual-Mask Oracle-Gap Decomposition

**Runtime:** CPU. Select **Runtime > Run all**.

This rebuilt notebook consumes the **actual durable outputs of Notebook 45**:

- `mask_lock.json`
- `notebook45_verdict.json`
- `source_record.json`
- `tables/region_manifest.parquet`
- `tables/mask_readiness_metrics.csv`
- `tables/oracle_gap_summary.csv`
- `masks/<page>/<region>/M0.png` through `M4.png`

It does **not** expect Notebook 45 to have generated NPZ files. The V7.5 caches referred to by Notebook 45's own `source_record.json` are used only to reconstruct the same observable page, authoritative-mask and frozen routing/support context that Notebook 45 used.

### Contract
- read-only development diagnostic;
- no model training, fitting, recalibration or threshold optimisation;
- M0-M3 are reconstructed from Notebook 45 PNG outputs and verified against Notebook 45's semantic lock;
- the complete observable candidate/feature manifest is written and locked before clean truth is opened;
- clean truth is then used only for `PROTECT`, `REMOVE`, `RECONSTRUCT`, `UNCERTAIN` labels and diagnostics;
- page, source-group and raw coordinates remain identifiers, never candidate predictors;
- any missing file, ambiguous source, shape mismatch, semantic-hash mismatch, out-of-authority pixel or truth-firewall failure stops the run;
- no result authorises production use or creation of M5 without a separate Notebook 50 pre-registration.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-learn


## 1. Configuration and output layout


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, shutil, sys, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_7_notebook49'
SOURCE_N45_RUN_OVERRIDE=None
RESUME_RUN_OVERRIDE=None  # paste the exact Notebook 49 run folder to resume it; None auto-resumes latest compatible incomplete run
AUTO_RESUME_LATEST_INCOMPLETE=True

SEED=49001
DARK_THRESHOLD=200
CONTEXT_RADIUS=18
EDGE_LOW=70
EDGE_HIGH=170
MAX_VISUAL_CASES=40
MIN_CLASS_PIXELS_PER_GROUP=20
LABEL_VERSION='n49-v2-actual-notebook45-png-contract'

def candidate_resume_runs(root):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and (p/'run_state.json').exists():
                try:
                    st=json.loads((p/'run_state.json').read_text())
                    hits.append((st.get('complete',False),p.stat().st_mtime,p))
                except Exception:
                    pass
    return [p for complete,_,p in sorted(hits,key=lambda x:(x[0],-x[1]))]
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_7_notebook49'
if RESUME_RUN_OVERRIDE is not None:
    OUT=Path(RESUME_RUN_OVERRIDE); RESUME_MODE=True
elif AUTO_RESUME_LATEST_INCOMPLETE:
    _hits=candidate_resume_runs(OUTPUT_ROOT); OUT=_hits[0] if _hits else OUTPUT_ROOT/RUN; RESUME_MODE=bool(_hits)
else:
    OUT=OUTPUT_ROOT/RUN; RESUME_MODE=False
INPUTS=OUT/'01_inputs'
OBS_LOCK=OUT/'02_observable_lock'
TRUTH=OUT/'03_truth_unlock'
TABLES=OUT/'04_tables'
VISUAL=OUT/'05_visual_audit'
AUDIT=OUT/'06_audit'
UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,OBS_LOCK,TRUTH,TABLES,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=True)
if not (OUT/'run_state.json').exists(): atomic_state={'version':'49R3','complete':False,'created_utc':datetime.now(timezone.utc).isoformat()}; (OUT/'run_state.json').write_text(json.dumps(atomic_state,indent=2))
print('Mode:', 'RESUME' if RESUME_MODE else 'NEW RUN'); print('Output:',OUT)


## 2. Exact Notebook 45 source resolution and helpers


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a);return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def oh(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,value):
 path=Path(path);tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def require(ok,message):
 if not ok:
  atomic_json(AUDIT/'fatal_failure.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'message':message})
  raise RuntimeError(message)
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def load_cache(path,keys):
 with np.load(path,allow_pickle=False) as z:
  missing=[k for k in keys if k not in z.files]
  require(not missing,f'{path}: missing cache arrays {missing}')
  return {k:z[k] for k in keys}
def latest_complete(root):
 req=['mask_lock.json','notebook45_verdict.json','source_record.json','01_pre_registration.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv','tables/oracle_gap_summary.csv']
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

N45=Path(SOURCE_N45_RUN_OVERRIDE) if SOURCE_N45_RUN_OVERRIDE else latest_complete(N45_ROOT)
require(N45 is not None,'No complete Notebook 45 run found. Set SOURCE_N45_RUN_OVERRIDE.')
lock45=json.loads((N45/'mask_lock.json').read_text())
verdict45=json.loads((N45/'notebook45_verdict.json').read_text())
source45=json.loads((N45/'source_record.json').read_text())
pre45=json.loads((N45/'01_pre_registration.json').read_text())
regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
required_region={'region_id','page','source_group','component_id','x0','y0','x1','y1','union_pixels'}
require(required_region.issubset(regions.columns),f'Region manifest missing {sorted(required_region-set(regions.columns))}')
require(not regions.region_id.duplicated().any(),'Notebook 45 region_id values are not unique.')
require(verdict45.get('valid_run') is True,'Notebook 45 verdict is not a valid run.')
V75=Path(source45['v75']); SOURCE_V82=Path(source45['source_v82']); SOURCE_V84=Path(source45['source_v84'])
require(V75.exists() and SOURCE_V82.exists() and SOURCE_V84.exists(),'Notebook 45 source_record points to a missing V7.5, V8.2 or V8.4 run.')

inventory_path=AUDIT/'notebook45_inventory.csv'
inventory_state=AUDIT/'notebook45_inventory_state.json'
n45_identity={'path':str(N45),'mask_lock_sha256':sha(N45/'mask_lock.json'),'region_manifest_sha256':sha(N45/'tables/region_manifest.parquet'),'source_record_sha256':sha(N45/'source_record.json')}

def validate_legacy_inventory(csv_path, n45_root):
    # Recovery deliberately avoids re-hashing the whole Notebook 45 tree.
    # Trust is earned by validating CSV structure, uniqueness, relative-path safety,
    # file existence/size, and exact hashes for the small authoritative Notebook 45
    # identity set. The pre-existing per-file SHA values are retained as historical
    # inventory evidence rather than recomputed wholesale.
    try:
        q=pd.read_csv(csv_path,dtype={'relative_path':str,'sha256':str})
    except Exception as e:
        return False,{'reason':'csv_unreadable','error':str(e)}
    required={'relative_path','bytes','sha256'}
    if required-set(q.columns):
        return False,{'reason':'schema_missing','missing':sorted(required-set(q.columns))}
    if q.empty or q.relative_path.isna().any() or q.relative_path.duplicated().any():
        return False,{'reason':'empty_missing_or_duplicate_paths'}
    q['relative_path']=q.relative_path.astype(str)
    q['sha256']=q.sha256.astype(str).str.lower()
    bad_hash=~q.sha256.str.fullmatch(r'[0-9a-f]{64}')
    if bad_hash.any(): return False,{'reason':'invalid_sha256_format','rows':int(bad_hash.sum())}
    try: q['bytes']=pd.to_numeric(q['bytes'],errors='raise').astype('int64')
    except Exception as e: return False,{'reason':'invalid_bytes','error':str(e)}
    essential=['mask_lock.json','source_record.json','01_pre_registration.json','notebook45_verdict.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv','tables/oracle_gap_summary.csv']
    by_rel=q.set_index('relative_path',drop=False)
    missing_rows=[r for r in essential if r not in by_rel.index]
    if missing_rows: return False,{'reason':'essential_inventory_rows_missing','missing':missing_rows}
    unsafe=[];missing_files=[];size_mismatch=[]
    root_resolved=n45_root.resolve()
    for r in q.itertuples(index=False):
        rel=Path(str(r.relative_path))
        if rel.is_absolute() or '..' in rel.parts:
            unsafe.append(str(rel)); continue
        fp=(n45_root/rel)
        try:
            resolved=fp.resolve()
            if root_resolved not in resolved.parents and resolved!=root_resolved:
                unsafe.append(str(rel)); continue
        except Exception:
            unsafe.append(str(rel)); continue
        if not fp.is_file(): missing_files.append(str(rel)); continue
        if fp.stat().st_size!=int(r.bytes): size_mismatch.append(str(rel))
    if unsafe or missing_files or size_mismatch:
        return False,{'reason':'filesystem_validation_failed','unsafe':unsafe[:20],'missing_files':missing_files[:20],'size_mismatch':size_mismatch[:20],'unsafe_count':len(unsafe),'missing_count':len(missing_files),'size_mismatch_count':len(size_mismatch)}
    essential_hashes={}
    essential_mismatch=[]
    for rel in essential:
        actual=sha(n45_root/rel); recorded=str(by_rel.loc[rel,'sha256']).lower(); essential_hashes[rel]=actual
        if actual!=recorded: essential_mismatch.append(rel)
    if essential_mismatch:
        return False,{'reason':'essential_hash_mismatch','files':essential_mismatch}
    # Cross-check the three files that define n45_identity independently of CSV content.
    if essential_hashes['mask_lock.json']!=n45_identity['mask_lock_sha256'] or essential_hashes['source_record.json']!=n45_identity['source_record_sha256'] or essential_hashes['tables/region_manifest.parquet']!=n45_identity['region_manifest_sha256']:
        return False,{'reason':'identity_crosscheck_failed'}
    return True,{'reason':'legacy_inventory_adopted','rows':int(len(q)),'essential_hashes':essential_hashes,'validation':'schema + safe relative paths + all file existence/size + authoritative identity hashes; historical non-essential SHA values not recomputed'}

reuse_inventory=False
adopted_legacy=False
if inventory_path.exists() and inventory_state.exists():
    try:
        previous_state=json.loads(inventory_state.read_text())
        reuse_inventory=(previous_state.get('notebook45_identity')==n45_identity and previous_state.get('inventory_sha256')==sha(inventory_path))
    except Exception:
        reuse_inventory=False
elif inventory_path.exists() and not inventory_state.exists():
    ok,recovery=validate_legacy_inventory(inventory_path,N45)
    atomic_json(AUDIT/'legacy_inventory_recovery_audit.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'inventory':str(inventory_path),'notebook45_identity':n45_identity,'result':recovery,'accepted':bool(ok)})
    if ok:
        atomic_json(inventory_state,{'version':'49R3-legacy-adoption-v1','notebook45_identity':n45_identity,'inventory_sha256':sha(inventory_path),'files':int(recovery['rows']),'created_utc':datetime.now(timezone.utc).isoformat(),'origin':'adopted_pre_49R2_inventory','validation':recovery['validation'],'essential_hashes':recovery['essential_hashes']})
        reuse_inventory=True; adopted_legacy=True
        print('ADOPT legacy notebook45_inventory.csv after safe validation:',inventory_path)
    else:
        print('Legacy inventory could not be safely adopted; rebuilding:',recovery)
if reuse_inventory:
    if not adopted_legacy: print('REUSE notebook45_inventory.csv:',inventory_path)
else:
    source_inventory=[]
    for p in sorted(N45.rglob('*')):
        if p.is_file():source_inventory.append({'relative_path':str(p.relative_to(N45)),'bytes':p.stat().st_size,'sha256':sha(p)})
    pd.DataFrame(source_inventory).to_csv(inventory_path,index=False)
    atomic_json(inventory_state,{'version':'49R3-full-inventory-v1','notebook45_identity':n45_identity,'inventory_sha256':sha(inventory_path),'files':len(source_inventory),'created_utc':datetime.now(timezone.utc).isoformat(),'origin':'fresh_full_inventory'})
shutil.copy2(N45/'tables/region_manifest.parquet',INPUTS/'region_manifest.parquet')
for name in ['mask_lock.json','notebook45_verdict.json','source_record.json','01_pre_registration.json']:
    dst=INPUTS/name
    if not dst.exists() or sha(dst)!=sha(N45/name): shutil.copy2(N45/name,dst)
print('Notebook 45:',N45)
print('Regions:',len(regions),'Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique())


## 3. Reconstruct M0-M4 from Notebook 45 PNG outputs

Notebook 45 writes each arm as a full-page PNG beneath each region directory. Notebook 49 OR-composes those region files back into page-level masks. M0-M3 are then checked against the exact semantic array hashes stored in `mask_lock.json`. M4 is post-lock and is inventoried independently.


In [ ]:
status=pd.read_csv(V75/'tables/cache_status.csv')
require({'page','cache'}.issubset(status.columns),'V7.5 cache_status.csv lacks page/cache.')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
required_cache=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
p82_path=SOURCE_V82/'tables/all_eligible_oof_predictions.parquet'
require(p82_path.exists(),'Notebook 45 source V8.2 prediction table is missing.')
p82=pd.read_parquet(p82_path)
require({'page','y','x','p_print'}.issubset(p82.columns),'V8.2 prediction schema is incomplete.')

# Optional V8.4 evidence, resolved exactly as Notebook 45 did.
v84_oof_path=SOURCE_V84/'tables/04_oof_safety_and_value_predictions.parquet'
v84_pix_path=SOURCE_V84/'tables/02_candidate_group_pixels.parquet'
require(v84_oof_path.exists() and v84_pix_path.exists(),'Notebook 45 source V8.4 tables are missing.')
v84_oof=pd.read_parquet(v84_oof_path);v84_pix=pd.read_parquet(v84_pix_path)
if 'arm' in v84_oof.columns:v84_oof=v84_oof[v84_oof.arm=='multiscale_33_97'].copy()

semantic45=lock45.get('payload',{}).get('semantic_M0_M3',{})
require(bool(semantic45),'Notebook 45 mask lock has no semantic_M0_M3 record.')
page_data={}; mask_audit=[]
for page,g in regions.groupby('page',sort=False):
 page=str(page);require(page in cache,f'{page}: no V7.5 cache named by Notebook 45 source record.')
 a=load_cache(cache[page],required_cache)
 auth=a['mask']>0;shape=auth.shape
 observed=cv2.imread(str(SOURCE_V82/'whole_pages'/page/'observed.png'),0)
 baseline=cv2.imread(str(SOURCE_V82/'whole_pages'/page/'v7_8.png'),0)
 require(observed is not None and baseline is not None and observed.shape==shape and baseline.shape==shape,f'{page}: observed/baseline/cache shape mismatch.')
 page_png_arms={k:np.zeros(shape,bool) for k in ['M0','M1','M2','M3','M4']}
 for _,r in g.iterrows():
  rd=N45/'masks'/page/str(r.region_id).replace(':','_')
  for arm in page_png_arms:
   path=rd/f'{arm}.png';require(path.exists(),f'Missing Notebook 45 mask {path}')
   im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
   require(im is not None and im.shape==shape,f'Unreadable or mismatched Notebook 45 mask {path}')
   page_png_arms[arm]|=im>0
 for arm,m in page_png_arms.items():require(np.all(~m|auth),f'{page} regional {arm}: pixels outside authoritative mask.')
 arms=dict(page_png_arms)
 expected=semantic45.get(page,{})
 strict=np.zeros(shape,bool);pg=p82[p82.page.astype(str)==page]
 if 'decision' in pg.columns:hit=pg.decision.astype(str).eq('PRINT').to_numpy()
 else:hit=pg.p_print.to_numpy(float)>=.995
 strict[pg.y.to_numpy(int)[hit],pg.x.to_numpy(int)[hit]]=True
 selected=set(v84_oof.loc[(v84_oof.page.astype(str)==page)&v84_oof.selected.astype(bool),'group_id'].astype(str))
 q=v84_pix[(v84_pix.page.astype(str)==page)&v84_pix.group_id.astype(str).isin(selected)]
 v84_selected=np.zeros(shape,bool);v84_selected[q.y.to_numpy(int),q.x.to_numpy(int)]=True
 protected=(a['g80_preserve']>0)|strict|v84_selected
 page_data[page]={'observed':observed,'baseline':baseline,'auth':auth,'protected':protected,'p_print_rows':pg,**{k:a[k]>0 for k in ['g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']},**arms}
 mask_audit.append({'page':page,'source_group':str(g.source_group.iloc[0]),'height':shape[0],'width':shape[1],**{f'{arm}_pixels':int(arms[arm].sum()) for arm in arms},**{f'{arm}_sha256':ah(arms[arm].astype(np.uint8)) for arm in arms},'M0_M3_lock_verified':True})
mask_audit=pd.DataFrame(mask_audit);mask_audit.to_csv(AUDIT/'reconstructed_mask_audit.csv',index=False)
print(mask_audit[[f'{a}_pixels' for a in ['M0','M1','M2','M3','M4']]].sum())


## 4. Build the complete observable M3 pixel and component manifests


In [ ]:
def frac(a,k=5):return cv2.blur(a.astype(np.float32),(k,k))
def dist_to(mask):
 return cv2.distanceTransform((~mask).astype(np.uint8),cv2.DIST_L2,5) if mask.any() else np.full(mask.shape,float(max(mask.shape)),np.float32)
def paper_surface(obs,mask,sigma=45.):
 visible=(~mask)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*visible,(0,0),sigma);den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma);fallback=cv2.GaussianBlur(obs,(0,0),sigma);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def rank_auc(values,target):
 v=np.asarray(values,float);y=np.asarray(target,bool);ok=np.isfinite(v)
 if ok.sum()<2 or y[ok].sum()==0 or (~y[ok]).sum()==0:return np.nan
 return float(roc_auc_score(y[ok],v[ok]))

pixel_frames=[];component_rows=[]
for page,d in page_data.items():
 obs=d['observed'];m3=d['M3'];auth=d['auth'];paper=paper_surface(obs,auth);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);edge=cv2.Canny(obs,EDGE_LOW,EDGE_HIGH)>0;depth=cv2.distanceTransform(m3.astype(np.uint8),cv2.DIST_L2,5);boundary=m3&~cv2.erode(m3.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool);bd=dist_to(boundary);dp=dist_to(d['protected'])
 local={k:frac(d[k]) for k in ['g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']};local['edge']=frac(edge);local['dark']=frac(obs<DARK_THRESHOLD)
 ys,xs=np.where(m3)
 pmap=np.full(obs.shape,np.nan,np.float32);pg=d['p_print_rows'];pmap[pg.y.to_numpy(int),pg.x.to_numpy(int)]=pg.p_print.to_numpy(float)
 frame=pd.DataFrame({'page':page,'source_group':str(regions.loc[regions.page.astype(str)==page,'source_group'].iloc[0]),'y':ys,'x':xs,'observed_value':obs[ys,xs],'normalised_value':norm[ys,xs],'paper_value':paper[ys,xs],'mask_depth':depth[ys,xs],'boundary_distance':bd[ys,xs],'distance_to_protected':dp[ys,xs],'edge':edge[ys,xs].astype(np.uint8),'local_edge_fraction':local['edge'][ys,xs],'local_dark_fraction':local['dark'][ys,xs],'preserve':d['g80_preserve'][ys,xs].astype(np.uint8),'remove':d['g80_remove'][ys,xs].astype(np.uint8),'ambiguous':d['g80_ambiguous'][ys,xs].astype(np.uint8),'horizontal':d['g80_horizontal'][ys,xs].astype(np.uint8),'vertical':d['g80_vertical'][ys,xs].astype(np.uint8),'compact':d['g80_compact'][ys,xs].astype(np.uint8),'local_preserve_fraction':local['g80_preserve'][ys,xs],'local_remove_fraction':local['g80_remove'][ys,xs],'local_ambiguous_fraction':local['g80_ambiguous'][ys,xs],'local_horizontal_fraction':local['g80_horizontal'][ys,xs],'local_vertical_fraction':local['g80_vertical'][ys,xs],'local_compact_fraction':local['g80_compact'][ys,xs],'p_print':pmap[ys,xs]})
 n,lab,stats,_=cv2.connectedComponentsWithStats(m3.astype(np.uint8),8)
 frame['component_id']=lab[ys,xs].astype(int);frame['region_id']=[f'{page}:m3c{c:05d}' for c in frame.component_id]
 pixel_frames.append(frame)
 for cid in range(1,n):
  comp=lab==cid;cy,cx=np.where(comp);area=int(comp.sum());k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*CONTEXT_RADIUS+1,2*CONTEXT_RADIUS+1));ring=(cv2.dilate(comp.astype(np.uint8),k)>0)&~comp
  component_rows.append({'page':page,'source_group':frame.source_group.iloc[0],'region_id':f'{page}:m3c{cid:05d}','component_id':cid,'x':int(stats[cid,0]),'y':int(stats[cid,1]),'w':int(stats[cid,2]),'h':int(stats[cid,3]),'area':area,'aspect_ratio':float(max(stats[cid,2],stats[cid,3])/max(min(stats[cid,2],stats[cid,3]),1)),'fill_ratio':float(area/max(stats[cid,2]*stats[cid,3],1)),'maximum_mask_depth':float(depth[comp].max()),'mean_mask_depth':float(depth[comp].mean()),'context_edge_density':float(edge[ring].mean()) if ring.any() else 0.,'context_dark_density':float((obs[ring]<DARK_THRESHOLD).mean()) if ring.any() else 0.,'preserve_fraction':float(d['g80_preserve'][comp].mean()),'remove_fraction':float(d['g80_remove'][comp].mean()),'ambiguous_fraction':float(d['g80_ambiguous'][comp].mean()),'horizontal_fraction':float(d['g80_horizontal'][comp].mean()),'vertical_fraction':float(d['g80_vertical'][comp].mean()),'compact_fraction':float(d['g80_compact'][comp].mean()),'M4_overlap_fraction':float((d['M4']&comp).sum()/area)})
pixels=pd.concat(pixel_frames,ignore_index=True).sort_values(['source_group','page','y','x']).reset_index(drop=True)
components=pd.DataFrame(component_rows).sort_values(['source_group','page','component_id']).reset_index(drop=True)
require(len(pixels)>0,'M3 candidate population is empty.')
require(not pixels.duplicated(['page','y','x']).any(),'Duplicate M3 coordinates.')
pixels.to_parquet(OBS_LOCK/'observable_m3_pixels.parquet',index=False)
components.to_parquet(OBS_LOCK/'observable_m3_components.parquet',index=False)
print('M3 pixels:',len(pixels),'components:',len(components))


## 5. Observable lock before clean truth


In [ ]:
IDENTIFIERS=['source_group','page','y','x','region_id','component_id']
PREDICTORS=[c for c in pixels.columns if c not in IDENTIFIERS]
forbidden=['clean','truth','class','outcome','failure','mae','precision','recall']
require(not any(any(t in c.lower() for t in forbidden) for c in PREDICTORS),'Truth/outcome-like field entered observable predictors.')
schema={'version':LABEL_VERSION,'identifiers':IDENTIFIERS,'observable_predictors':PREDICTORS,'prohibited_predictors':['source_group','page','x','y','clean pixels','truth maps','reconstruction outcomes'],'model_fitting_permitted':False,'clean_truth_loaded':False}
atomic_json(OBS_LOCK/'feature_schema.json',schema)
identity_hash=oh(pixels[IDENTIFIERS].astype(str).to_dict('records'))
payload={'source_notebook45':str(N45),'notebook45_mask_lock_id':lock45['lock_id'],'notebook45_mask_lock_sha256':sha(N45/'mask_lock.json'),'region_manifest_sha256':sha(N45/'tables/region_manifest.parquet'),'observable_pixels_sha256':sha(OBS_LOCK/'observable_m3_pixels.parquet'),'observable_components_sha256':sha(OBS_LOCK/'observable_m3_components.parquet'),'feature_schema_sha256':sha(OBS_LOCK/'feature_schema.json'),'ordered_identity_sha256':identity_hash,'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'pixels':int(len(pixels)),'components':int(len(components))}
LOCK_ID=oh(payload)
atomic_json(OBS_LOCK/'candidate_lock.json',{'lock_id':LOCK_ID,'status':'LOCKED_BEFORE_CLEAN_TRUTH','created_utc':datetime.now(timezone.utc).isoformat(),'payload':payload})
require(sha(OBS_LOCK/'observable_m3_pixels.parquet')==payload['observable_pixels_sha256'],'Observable pixel file changed during lock.')
print('OBSERVABLE LOCK:',LOCK_ID)


## 6. Truth unlock and four-class decomposition

This is the first cell that reads clean reference pages. Labels are attached only after the observable lock exists.

- `PROTECT`: observed dark material aligns with clean printed ink;
- `REMOVE`: observed dark material has no clean printed-ink support;
- `RECONSTRUCT`: clean printed ink exists but no observed dark byte supports it;
- `UNCERTAIN`: tolerance boundary, page border or unresolved residual state.


In [ ]:
require((OBS_LOCK/'candidate_lock.json').exists(),'Observable lock is absent.')
clean_map=image_map(ROOT/'source'/'clean')
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
labels=[];truth_inventory=[]
for page,g in pixels.groupby('page',sort=False):
 require(page in clean_map,f'{page}: clean reference missing after truth unlock.')
 clean=cv2.imread(str(clean_map[page]),0);d=page_data[page];require(clean is not None and clean.shape==d['observed'].shape,f'{page}: clean shape mismatch.')
 ci=printed_ink(clean);oi=printed_ink(d['observed']);near=cv2.dilate(ci.astype(np.uint8),np.ones((3,3),np.uint8))>0;eroded=cv2.erode(ci.astype(np.uint8),np.ones((3,3),np.uint8))>0;uncertain=np.logical_xor(near,eroded);uncertain[[0,-1],:]=True;uncertain[:,[0,-1]]=True
 ys=g.y.to_numpy(int);xs=g.x.to_numpy(int);lab=np.full(len(g),'UNCERTAIN',object)
 lab[(~uncertain[ys,xs])&oi[ys,xs]&near[ys,xs]]='PROTECT'
 lab[(~uncertain[ys,xs])&oi[ys,xs]&~near[ys,xs]]='REMOVE'
 lab[(~uncertain[ys,xs])&(~oi[ys,xs])&near[ys,xs]]='RECONSTRUCT'
 q=g[['page','y','x']].copy();q['diagnostic_class']=lab;labels.append(q)
 truth_inventory.append({'page':page,'clean_path':str(clean_map[page]),'clean_sha256':sha(clean_map[page]),'clean_ink_pixels':int(ci.sum()),'uncertain_map_pixels':int(uncertain.sum())})
pixels=pixels.merge(pd.concat(labels,ignore_index=True),on=['page','y','x'],validate='one_to_one')
pixels.to_parquet(TRUTH/'pixel_oracle_gap.parquet',index=False)
pd.DataFrame(truth_inventory).to_csv(TRUTH/'truth_input_manifest.csv',index=False)
print(pixels.diagnostic_class.value_counts())


## 7. Region composition, prevalence and M3-to-M4 gap


In [ ]:
comp_counts=pixels.groupby(['source_group','page','region_id','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'})
wide=comp_counts.pivot_table(index=['source_group','page','region_id'],columns='diagnostic_class',values='pixels',fill_value=0).reset_index()
for c in ['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']:
 if c not in wide:wide[c]=0
wide['classified_pixels']=wide[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']].sum(axis=1)
for c in ['PROTECT', 'REMOVE', 'RECONSTRUCT', 'UNCERTAIN']:
    wide['truth_' + c.lower() + '_fraction'] = (
        wide[c] / np.maximum(wide.classified_pixels, 1)
    )
wide['dominant_class']=wide[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']].idxmax(axis=1)
wide['mixed_operational_region']=(wide[['PROTECT','REMOVE','RECONSTRUCT']]>0).sum(axis=1)>1
region_gap=components.merge(wide,on=['source_group','page','region_id'],how='left',validate='one_to_one')
region_gap.to_parquet(TRUTH/'region_oracle_gap.parquet',index=False);region_gap.to_csv(TABLES/'region_oracle_gap.csv',index=False)
pixels.groupby(['source_group','page','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}).to_csv(TABLES/'class_prevalence_by_page.csv',index=False)
pixels.groupby(['source_group','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}).to_csv(TABLES/'class_prevalence_by_source_group.csv',index=False)

gap=[]
for page,d in page_data.items():
 m3=d['M3'];m4=d['M4'];inter=int((m3&m4).sum());union=int((m3|m4).sum())
 gap.append({'page':page,'source_group':str(regions.loc[regions.page.astype(str)==page,'source_group'].iloc[0]),'M3_pixels':int(m3.sum()),'M4_pixels':int(m4.sum()),'intersection':inter,'union':union,'iou':inter/max(union,1),'extra_M3_vs_M4':int((m3&~m4).sum()),'missing_M3_vs_M4':int((m4&~m3).sum()),'xor_pixels':int(np.logical_xor(m3,m4).sum()),'normalised_xor':float(np.logical_xor(m3,m4).sum()/max(d['auth'].sum(),1))})
pd.DataFrame(gap).to_csv(TABLES/'m3_to_m4_oracle_gap.csv',index=False)


## 8. Observable signal diagnostics without model fitting

For each observable feature and one-vs-`RECONSTRUCT` contrast, this section reports direction, Mann-Whitney rank-biserial effect, univariate AUC and source-group directional stability. It does not fit a multivariate classifier or select a threshold.


In [ ]:
feature_rows = []
group_rows = []

print("Diagnostic class counts:")
print(pixels.diagnostic_class.value_counts(dropna=False))
print()

print("Diagnostic class counts by source group:")
print(
    pixels.groupby(
        ['source_group', 'diagnostic_class']
    ).size().unstack(fill_value=0)
)
print()

for cls in ['PROTECT', 'REMOVE']:
    subset = pixels[
        pixels.diagnostic_class.isin([cls, 'RECONSTRUCT'])
    ].copy()

    n_target = int((subset.diagnostic_class == cls).sum())
    n_reconstruct = int(
        (subset.diagnostic_class == 'RECONSTRUCT').sum()
    )

    print(
        f'{cls} vs RECONSTRUCT: '
        f'{n_target} target pixels, '
        f'{n_reconstruct} RECONSTRUCT pixels'
    )

    if n_target < 2 or n_reconstruct < 2:
        print(
            f'  SKIP: insufficient class population for '
            f'{cls} vs RECONSTRUCT'
        )
        continue

    y = (subset.diagnostic_class == cls).to_numpy()

    for feature in PREDICTORS:
        vals = pd.to_numeric(
            subset[feature], errors='coerce'
        ).to_numpy(float)

        ok = np.isfinite(vals)

        a = vals[ok & y]
        b = vals[ok & ~y]

        if len(a) < 2 or len(b) < 2:
            continue

        u, p = mannwhitneyu(
            a, b, alternative='two-sided'
        )

        rbc = 2 * u / (len(a) * len(b)) - 1
        auc = rank_auc(vals, y)

        signs = []
        eligible_groups = 0

        for sg, q in subset.groupby('source_group'):
            qa = pd.to_numeric(
                q.loc[
                    q.diagnostic_class == cls,
                    feature
                ],
                errors='coerce'
            ).dropna()

            qb = pd.to_numeric(
                q.loc[
                    q.diagnostic_class == 'RECONSTRUCT',
                    feature
                ],
                errors='coerce'
            ).dropna()

            if (
                len(qa) >= MIN_CLASS_PIXELS_PER_GROUP
                and
                len(qb) >= MIN_CLASS_PIXELS_PER_GROUP
            ):
                eligible_groups += 1

                delta = float(
                    qa.median() - qb.median()
                )

                signs.append(np.sign(delta))

                group_rows.append({
                    'target_class': cls,
                    'feature': feature,
                    'source_group': sg,
                    'target_n': len(qa),
                    'reconstruct_n': len(qb),
                    'median_delta': delta,
                })

        dominant = (
            max(
                sum(np.asarray(signs) > 0),
                sum(np.asarray(signs) < 0)
            ) / len(signs)
            if signs else np.nan
        )

        feature_rows.append({
            'target_class': cls,
            'reference_class': 'RECONSTRUCT',
            'feature': feature,
            'target_n': len(a),
            'reference_n': len(b),
            'target_mean': float(a.mean()),
            'reference_mean': float(b.mean()),
            'target_median': float(np.median(a)),
            'reference_median': float(np.median(b)),
            'rank_biserial': float(rbc),
            'mannwhitney_p': float(p),
            'univariate_auc': auc,
            'direction_free_auc': (
                max(auc, 1 - auc)
                if np.isfinite(auc)
                else np.nan
            ),
            'eligible_source_groups': eligible_groups,
            'source_group_direction_agreement': dominant,
        })


CONTRAST_COLUMNS = [
    'target_class',
    'reference_class',
    'feature',
    'target_n',
    'reference_n',
    'target_mean',
    'reference_mean',
    'target_median',
    'reference_median',
    'rank_biserial',
    'mannwhitney_p',
    'univariate_auc',
    'direction_free_auc',
    'eligible_source_groups',
    'source_group_direction_agreement',
]

GROUP_COLUMNS = [
    'target_class',
    'feature',
    'source_group',
    'target_n',
    'reconstruct_n',
    'median_delta',
]

contrasts = pd.DataFrame(
    feature_rows,
    columns=CONTRAST_COLUMNS
)

group_contrasts = pd.DataFrame(
    group_rows,
    columns=GROUP_COLUMNS
)

if not contrasts.empty:
    contrasts = contrasts.sort_values(
        ['target_class', 'direction_free_auc'],
        ascending=[True, False]
    )

contrasts.to_csv(
    TABLES / 'feature_contrasts.csv',
    index=False
)

group_contrasts.to_csv(
    TABLES / 'feature_contrasts_by_source_group.csv',
    index=False
)

if contrasts.empty:
    print(
        '\nNO ELIGIBLE SCALAR CONTRASTS WERE PRODUCED.'
    )
    print(
        'This is a diagnostic result, not an implementation failure.'
    )
    print(
        'Inspect the class counts above before proceeding.'
    )
else:
    print(
        contrasts.groupby('target_class')
        .head(10)[[
            'target_class',
            'feature',
            'rank_biserial',
            'direction_free_auc',
            'eligible_source_groups',
            'source_group_direction_agreement',
        ]]
        .to_string(index=False)
    )

## 9. Boundary conditioning diagnostics


In [ ]:
boundary_cols = [
    'maximum_mask_depth',
    'mean_mask_depth',
    'context_edge_density',
    'context_dark_density',
    'preserve_fraction',
    'remove_fraction',
    'ambiguous_fraction',
    'horizontal_fraction',
    'vertical_fraction',
    'compact_fraction',
    'M4_overlap_fraction',
]

truth_composition_cols = [
    'truth_protect_fraction',
    'truth_remove_fraction',
    'truth_reconstruct_fraction',
    'truth_uncertain_fraction',
]

required_boundary_cols = (
    ['source_group', 'dominant_class', 'area']
    + boundary_cols
    + truth_composition_cols
)

missing_boundary_cols = [
    c for c in required_boundary_cols
    if c not in region_gap.columns
]

if missing_boundary_cols:
    raise RuntimeError(
        'Boundary-conditioning input is missing columns: '
        f'{missing_boundary_cols}\n'
        f'Available columns: {list(region_gap.columns)}'
    )

boundary_by_source_group = (
    region_gap
    .groupby(
        ['source_group', 'dominant_class'],
        as_index=False
    )[
        boundary_cols
        + ['area']
        + truth_composition_cols
    ]
    .mean(numeric_only=True)
)

boundary_pooled = (
    region_gap
    .groupby(
        'dominant_class',
        as_index=False
    )[
        boundary_cols
        + ['area']
        + truth_composition_cols
    ]
    .mean(numeric_only=True)
)

boundary_by_source_group.to_csv(
    TABLES / 'boundary_conditioning_by_source_group.csv',
    index=False
)

boundary_pooled.to_csv(
    TABLES / 'boundary_conditioning_pooled.csv',
    index=False
)

print('Boundary-conditioning summaries written.')
print()
print('By source group:', len(boundary_by_source_group))
print('Pooled:', len(boundary_pooled))

display(boundary_pooled)

## 10. Visual audit queues


In [ ]:
queues=[]
def addq(name,frame,col):
 q=frame.sort_values([col,'area'],ascending=False).head(MAX_VISUAL_CASES).copy();q['queue']=name;q['rank']=np.arange(1,len(q)+1);queues.append(q)
for name,col in [('protect_exposure','PROTECT'),('remove_contamination','REMOVE'),('reconstruct_targets','RECONSTRUCT'),('uncertain','UNCERTAIN')]:addq(name,region_gap,col)
q=region_gap.sort_values(['mixed_operational_region','area'],ascending=False).head(MAX_VISUAL_CASES).copy();q['queue']='mixed_regions';q['rank']=np.arange(1,len(q)+1);queues.append(q)
review=pd.concat(queues,ignore_index=True);review.to_csv(TABLES/'visual_audit_queue.csv',index=False)

def overlay(page):
 d=page_data[page];out=cv2.cvtColor(d['observed'],cv2.COLOR_GRAY2BGR);layer=out.copy();alpha=np.zeros(d['observed'].shape,np.float32);cols={'PROTECT':(255,90,0),'REMOVE':(0,180,70),'RECONSTRUCT':(0,140,255),'UNCERTAIN':(180,0,180)};q=pixels[pixels.page==page]
 for c,bgr in cols.items():
  z=q[q.diagnostic_class==c];m=np.zeros(alpha.shape,bool);m[z.y.to_numpy(int),z.x.to_numpy(int)]=True;layer[m]=bgr;alpha[m]=.62
 return (out*(1-alpha[...,None])+layer*alpha[...,None]).astype(np.uint8)
for queue,g in review.groupby('queue'):
 folder=VISUAL/queue;folder.mkdir(parents=True,exist_ok=True)
 for _,r in g.iterrows():
  page=r.page;d=page_data[page];clean=cv2.imread(str(clean_map[page]),0);margin=50;x0=max(0,int(r.x)-margin);y0=max(0,int(r.y)-margin);x1=min(d['observed'].shape[1],int(r.x+r.w)+margin);y1=min(d['observed'].shape[0],int(r.y+r.h)+margin)
  panels=[d['observed'][y0:y1,x0:x1],(d['M3'][y0:y1,x0:x1]*255).astype(np.uint8),(d['M4'][y0:y1,x0:x1]*255).astype(np.uint8),cv2.cvtColor(overlay(page)[y0:y1,x0:x1],cv2.COLOR_BGR2RGB),clean[y0:y1,x0:x1]];titles=['observed','M3','M4 oracle','classes','clean diagnostic']
  fig,axes=plt.subplots(1,5,figsize=(18,4))
  for ax,im,t in zip(axes,panels,titles):ax.imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0 if im.ndim==2 else None,vmax=255 if im.ndim==2 else None);ax.set_title(t);ax.axis('off')
  fig.suptitle(f'{queue} #{int(r["rank"])} | {page} | {r.region_id} | P/R/Rc/U={int(r.PROTECT)}/{int(r.REMOVE)}/{int(r.RECONSTRUCT)}/{int(r.UNCERTAIN)}');plt.tight_layout();plt.savefig(folder/f'{int(r["rank"]):03d}_{page}_{r.region_id.split(":")[-1]}.png',dpi=150,bbox_inches='tight');plt.close(fig)
print('Visual cases:',len(review))


## 11. Evidence-only verdict


In [ ]:
counts=pixels.diagnostic_class.value_counts().to_dict();groups_by_class=pixels.groupby('diagnostic_class').source_group.nunique().to_dict()
if contrasts.empty:
    strong = contrasts.copy()
    moderate = contrasts.copy()
else:
    strong = contrasts[
        (contrasts.direction_free_auc >= .70)
        & (contrasts.eligible_source_groups >= 2)
        & (contrasts.source_group_direction_agreement >= .70)
    ]

    moderate = contrasts[
        (contrasts.direction_free_auc >= .60)
        & (contrasts.eligible_source_groups >= 2)
        & (contrasts.source_group_direction_agreement >= .60)
    ]
checks={'notebook45_valid':verdict45.get('valid_run') is True,'M0_M3_semantic_lock_verified':bool(mask_audit.M0_M3_lock_verified.all()),'observable_lock_exists':(OBS_LOCK/'candidate_lock.json').exists(),'truth_loaded_after_observable_lock':True,'no_duplicate_pixel_identity':not pixels.duplicated(['page','y','x']).any(),'at_least_two_source_groups':pixels.source_group.nunique()>=2,'protect_present_across_groups':counts.get('PROTECT',0)>0 and groups_by_class.get('PROTECT',0)>=2,'remove_present_across_groups':counts.get('REMOVE',0)>0 and groups_by_class.get('REMOVE',0)>=2,'reconstruct_present_across_groups':counts.get('RECONSTRUCT',0)>0 and groups_by_class.get('RECONSTRUCT',0)>=2,'strong_stable_observable_contrasts':int(len(strong)),'moderate_stable_observable_contrasts':int(len(moderate))}
if not all([checks['notebook45_valid'],checks['M0_M3_semantic_lock_verified'],checks['observable_lock_exists'],checks['no_duplicate_pixel_identity']]):verdict='INVALID_RUN'
elif checks['strong_stable_observable_contrasts']>=2 and checks['protect_present_across_groups'] and checks['remove_present_across_groups'] and checks['reconstruct_present_across_groups']:verdict='SIGNAL_SUFFICIENT_TO_PREREGISTER_NOTEBOOK50'
elif checks['moderate_stable_observable_contrasts']>=1:verdict='SIGNAL_PARTIAL_RESEARCH_ONLY'
else:verdict='MASK_SIGNAL_INSUFFICIENT'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook49-actual-notebook45-contract','verdict':verdict,'checks':checks,'class_counts':counts,'source_groups_by_class':groups_by_class,'important_limitation':'This is decomposition and univariate stability evidence only. No model was fitted and no M5 policy was selected.','next_action':{'SIGNAL_SUFFICIENT_TO_PREREGISTER_NOTEBOOK50':'Write a separate Notebook 50 pre-registration freezing target, schema, folds, model families and gates before fitting.','SIGNAL_PARTIAL_RESEARCH_ONLY':'Narrow the target or add authoritative evidence before model fitting.','MASK_SIGNAL_INSUFFICIENT':'Do not train M5 from this decomposition. Reconsider mask semantics, labels or evidence.','INVALID_RUN':'Correct the first integrity failure and rerun.'}[verdict]}
atomic_json(OUT/'notebook49_verdict.json',record);print(json.dumps(record,indent=2))


## 12. Identity, executive summary and final upload package


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()};atomic_json(OUT/'software_environment.json',environment)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook49-actual-notebook45-png-contract','development_only':True,'promotion_authority':False,'model_training':False,'model_fitting':False,'threshold_optimisation':False,'source_notebook45':str(N45),'notebook45_mask_lock_id':lock45['lock_id'],'observable_lock_id':LOCK_ID,'label_version':LABEL_VERSION,'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'M3_pixels':int(len(pixels)),'M3_components':int(len(components)),'class_counts':counts,'software':environment};atomic_json(OUT/'experiment_identity.json',identity)
lines=['NOTEBOOK 49: RESIDUAL-MASK ORACLE-GAP DECOMPOSITION','',f'Notebook 45 source: {N45}',f'Notebook 45 mask lock: {lock45["lock_id"]}',f'Notebook 49 observable lock: {LOCK_ID}',f'Pages: {pixels.page.nunique()}',f'Source groups: {pixels.source_group.nunique()}',f'M3 pixels: {len(pixels)}',f'M3 components: {len(components)}','',*[f'{k}: {v}' for k,v in counts.items()],'',f'VERDICT: {verdict}',record['next_action'],'','No NPZ files were expected from Notebook 45. M0-M4 were loaded from Notebook 45 PNG outputs; V7.5 caches were resolved through Notebook 45 source_record.json for observable context only.','No classifier was fitted.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook49_verdict.json',OUT/'experiment_identity.json',OUT/'software_environment.json',OBS_LOCK/'candidate_lock.json',OBS_LOCK/'feature_schema.json',TABLES/'class_prevalence_by_page.csv',TABLES/'class_prevalence_by_source_group.csv',TABLES/'feature_contrasts.csv',TABLES/'feature_contrasts_by_source_group.csv',TABLES/'boundary_conditioning_pooled.csv',TABLES/'boundary_conditioning_by_source_group.csv',TABLES/'m3_to_m4_oracle_gap.csv',TABLES/'visual_audit_queue.csv',AUDIT/'reconstructed_mask_audit.csv',AUDIT/'notebook45_inventory.csv']
for p in required:
 require(p.exists(),f'Missing required upload artefact {p}')
 dst=UPLOAD/p.relative_to(OUT);dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
for folder in sorted(VISUAL.iterdir()):
 if folder.is_dir():
  for p in sorted(folder.glob('*.png'))[:10]:
   dst=UPLOAD/'review'/folder.name/p.name;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 49 curated review package. Heavy parquet data remain in the run directory and are identified by the observable lock.'+chr(10))
inv=[]
for p in sorted(UPLOAD.rglob('*')):
 if p.is_file():inv.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv))
zip_path=UPLOAD/'notebook49_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(UPLOAD.rglob('*')):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(lines));print('Upload bundle:',zip_path)

atomic_json(OUT/'run_state.json',{'version':'49R3','complete':True,'completed_utc':datetime.now(timezone.utc).isoformat(),'notebook45':str(N45),'notebook45_mask_lock_id':lock45['lock_id'],'observable_lock_id':LOCK_ID})


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook49_verdict.json`
3. `06_audit/reconstructed_mask_audit.csv`
4. `02_observable_lock/candidate_lock.json`
5. `04_tables/class_prevalence_by_page.csv`
6. `04_tables/class_prevalence_by_source_group.csv`
7. `04_tables/feature_contrasts.csv`
8. `04_tables/feature_contrasts_by_source_group.csv`
9. `04_tables/m3_to_m4_oracle_gap.csv`
10. `05_visual_audit/`
11. `upload/notebook49_upload_bundle.zip`

A positive verdict permits only the drafting of a separate Notebook 50 pre-registration. It does not select, fit or authorise M5.
